# Session 13: Network Security

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-13-network-security.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-13_firewall_acl.py`

In [1]:
"""Session 13 - A first-match packet-filter (ACL / firewall rule) evaluator.

Rules are checked top-down; the first rule that matches decides. If nothing
matches, the implicit "deny any" at the end applies (as on Cisco ACLs).
Run:  python session-13_firewall_acl.py
"""
import ipaddress

RULES = [  # action, protocol, source, destination, destination port (None = any), note
    ("permit", "tcp", "any",          "10.1.5.20/32", 443,  "public web server HTTPS"),
    ("permit", "tcp", "any",          "10.1.5.20/32", 80,   "HTTP (redirects to HTTPS)"),
    ("permit", "tcp", "10.1.9.0/24",  "10.1.5.0/24",  22,   "admin subnet SSH"),
    ("deny",   "tcp", "any",          "any",          3389, "no RDP from anywhere"),
    ("permit", "udp", "10.1.0.0/16",  "10.1.0.53/32", 53,   "internal DNS"),
    ("permit", "icmp", "10.1.0.0/16", "any",          None, "internal ping"),
]


def match(net, ip):
    return net == "any" or ipaddress.ip_address(ip) in ipaddress.ip_network(net)


def evaluate(proto, src, dst, port):
    for i, (act, p, s, d, dp, note) in enumerate(RULES, 1):
        if p == proto and match(s, src) and match(d, dst) and (dp is None or dp == port):
            return act.upper(), f"rule {i}: {note}"
    return "DENY", "implicit deny (no rule matched)"


if __name__ == "__main__":
    packets = [
        ("tcp", "198.51.100.7", "10.1.5.20", 443),
        ("tcp", "198.51.100.7", "10.1.5.20", 22),
        ("tcp", "10.1.9.15",    "10.1.5.20", 22),
        ("tcp", "10.1.9.15",    "10.1.5.30", 3389),
        ("udp", "10.1.7.7",     "10.1.0.53", 53),
        ("udp", "198.51.100.7", "10.1.0.53", 53),
        ("icmp", "10.1.7.7",    "8.8.8.8",   None),
        ("tcp", "10.1.7.7",     "10.1.5.20", 8080),
    ]
    for proto, s, d, p in packets:
        verdict, why = evaluate(proto, s, d, p)
        print(f"{proto:<4} {s:>14} -> {d:<10}:{str(p or '-'):<5} {verdict:<7} {why}")

tcp    198.51.100.7 -> 10.1.5.20 :443   PERMIT  rule 1: public web server HTTPS
tcp    198.51.100.7 -> 10.1.5.20 :22    DENY    implicit deny (no rule matched)
tcp       10.1.9.15 -> 10.1.5.20 :22    PERMIT  rule 3: admin subnet SSH
tcp       10.1.9.15 -> 10.1.5.30 :3389  DENY    rule 4: no RDP from anywhere
udp        10.1.7.7 -> 10.1.0.53 :53    PERMIT  rule 5: internal DNS
udp    198.51.100.7 -> 10.1.0.53 :53    DENY    implicit deny (no rule matched)
icmp       10.1.7.7 -> 8.8.8.8   :-     PERMIT  rule 6: internal ping
tcp        10.1.7.7 -> 10.1.5.20 :8080  DENY    implicit deny (no rule matched)


## `session-13_ids_detect.py`

In [2]:
"""Session 13 - Simple intrusion-detection rules over a connection log.

Detects three classic patterns in a recorded lab log:
  * port scan    : one source touches many different ports in a short time
  * brute force  : many failed logins for one account
  * data exfil   : unusually large outbound transfer
Run:  python session-13_ids_detect.py
"""
from collections import defaultdict

LOG = """\
09:00:01 10.1.7.7 10.1.5.20 443 ok 12000
09:00:02 10.1.7.8 10.1.5.20 443 ok 9000
09:01:10 10.1.66.6 10.1.5.30 21 refused 0
09:01:10 10.1.66.6 10.1.5.30 22 refused 0
09:01:11 10.1.66.6 10.1.5.30 23 refused 0
09:01:11 10.1.66.6 10.1.5.30 25 refused 0
09:01:11 10.1.66.6 10.1.5.30 80 ok 0
09:01:12 10.1.66.6 10.1.5.30 139 refused 0
09:01:12 10.1.66.6 10.1.5.30 445 refused 0
09:01:12 10.1.66.6 10.1.5.30 3389 refused 0
09:05:00 10.1.8.4 10.1.5.20 22 authfail 0
09:05:03 10.1.8.4 10.1.5.20 22 authfail 0
09:05:06 10.1.8.4 10.1.5.20 22 authfail 0
09:05:09 10.1.8.4 10.1.5.20 22 authfail 0
09:05:12 10.1.8.4 10.1.5.20 22 authfail 0
09:05:15 10.1.8.4 10.1.5.20 22 authfail 0
09:20:00 10.1.5.44 203.0.113.99 443 ok 2400000000
"""

SCAN_PORTS, SCAN_WINDOW = 5, 10        # >= 5 distinct ports within 10 s
BRUTE_FAILS = 5
EXFIL_BYTES = 500_000_000


def sec(t):
    h, m, s = map(int, t.split(":")); return h * 3600 + m * 60 + s


if __name__ == "__main__":
    ports = defaultdict(list); fails = defaultdict(int); alerts = []
    for line in LOG.strip().splitlines():
        t, src, dst, port, status, nbytes = line.split()
        ports[(src, dst)].append((sec(t), int(port)))
        recent = {p for ts, p in ports[(src, dst)] if sec(t) - ts <= SCAN_WINDOW}
        if len(recent) == SCAN_PORTS:
            alerts.append(f"{t} PORT SCAN   {src} -> {dst}: {sorted(recent)}")
        if status == "authfail":
            fails[(src, dst)] += 1
            if fails[(src, dst)] == BRUTE_FAILS:
                alerts.append(f"{t} BRUTE FORCE {src} -> {dst}:{port} {BRUTE_FAILS} failures")
        if int(nbytes) > EXFIL_BYTES:
            alerts.append(f"{t} LARGE UPLOAD {src} -> {dst}: {int(nbytes) / 1e9:.1f} GB")
    for a in alerts:
        print(a)
    print(f"\n{len(alerts)} alerts from {len(LOG.splitlines())} log lines")

09:01:11 PORT SCAN   10.1.66.6 -> 10.1.5.30: [21, 22, 23, 25, 80]
09:05:12 BRUTE FORCE 10.1.8.4 -> 10.1.5.20:22 5 failures
09:20:00 LARGE UPLOAD 10.1.5.44 -> 203.0.113.99: 2.4 GB

3 alerts from 17 log lines
